# VEDA AI — Intelligence Pipeline Interactive Notebook

This notebook provides live, interactive access to the core intelligence components:
1. **Entity Extraction** — Extract industrial entities from text
2. **Chunk Builder** — Split documents into semantic chunks
3. **RAG Pipeline** — Full retrieval-augmented generation pipeline
4. **Knowledge Graph** — Query the Apache AGE knowledge graph

Run this notebook alongside the FastAPI server for interactive exploration.

In [ ]:
import sys
sys.path.insert(0, '/app')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
print('VEDA AI Intelligence Notebook initialized.')

## 1. Entity Extraction

Extract industrial entities (equipment tags, measurements, regulations, failure modes, etc.) from raw text.

In [ ]:
from src.entity_extraction.extractor import extractor

sample_text = """
Pump PUMP-101 at Tank Farm A showed bearing seizure during Q1 2025 inspection.
The operating temperature was 350°C and pressure at 15.5 kg/cm².
Work Order WO-2024-1423 raised. Maintenance: bearing replacement.
Refer to OISD-154 for safety. Material: SS316. Chemical: Crude oil.
"""

entities = extractor.extract_entities(sample_text)

print(f'Extracted {len(entities)} entities:\n')
for e in entities:
    print(f'  [{e.entity_type}] {e.value} (confidence: {e.confidence:.2f})')

## 2. Chunk Builder

Split a document into semantic chunks with token estimation.

In [ ]:
from src.knowledge_graph.chunk_builder import ChunkBuilder

builder = ChunkBuilder(max_chunk_size=500, chunk_overlap=100)

chunks = builder.split_into_chunks(sample_text, document_id='demo-doc-001')

print(f'Split into {len(chunks)} chunks:\n')
for i, chunk in enumerate(chunks):
    print(f'  Chunk {i}: {chunk.token_estimate} tokens, {len(chunk.text)} chars')
    print(f'    Preview: {chunk.text[:120]}...\n')

## 3. Embedding Model

Verify the embedding model loads correctly and generates normalized vectors.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

test_queries = [
    'What is the maintenance procedure for PUMP-101?',
    'Show bearing seizure failure history',
    'List all equipment in Tank Farm A',
]

embeddings = model.encode(test_queries, normalize_embeddings=True)

print(f'Model: {model.get_sentence_embedding_dimension()}-dimensional embeddings\n')
for q, emb in zip(test_queries, embeddings):
    norm = np.linalg.norm(emb)
    print(f'  Query: "{q}"')
    print(f'    Shape: {emb.shape}, L2 norm: {norm:.4f}\n')

## 4. RAG Pipeline (Prepare Context)

Run the full RAG pipeline to prepare context for an LLM.

> **Note:** This requires PostgreSQL, OpenSearch, and Apache AGE to be running.
> Use `docker compose up -d` to start infrastructure services.

In [ ]:
# Uncomment to run against live infrastructure:
# from src.rag.pipeline import rag_pipeline
# import asyncio
#
# context = asyncio.get_event_loop().run_until_complete(
#     rag_pipeline.prepare(
#         query='What is the maintenance history of PUMP-101?',
#         tenant_id='your-tenant-id',
#     )
# )
#
# print(f'Prepared {len(context.chunks)} chunks')
# print(f'Total tokens: {context.total_token_estimate}')
# print(f'Entities found: {len(context.entities_found)}')
# print(f'\nSystem Prompt:\n{context.system_prompt[:200]}...')
# print(f'\nFormatted Prompt:\n{context.formatted_prompt[:500]}...')

print('RAG pipeline ready. Uncomment the cell above and provide a tenant_id to test.')

## 5. Prompt Formatting Demo

Build a formatted LLM prompt from chunks and graph context.

In [ ]:
from src.knowledge_graph.chunk_builder import ChunkBuilder, PreparedChunk

builder = ChunkBuilder()

demo_chunks = [
    PreparedChunk(
        text='PUMP-101 bearing replacement completed on 23-Mar-2024 by Rajesh Kumar.',
        source_type='vector',
        relevance_score=0.92,
        document_id='doc-001',
        metadata={'title': 'Maintenance Report Q1-2025'},
        token_estimate=builder.estimate_tokens('PUMP-101 bearing replacement completed on 23-Mar-2024.'),
        entities_in_chunk=[{'value': 'PUMP-101', 'type': 'EQUIPMENT_TAG', 'normalized_value': 'PUMP-101', 'confidence': 0.95}],
        graph_context=[],
    ),
]

prompt = builder.build_formatted_prompt(
    query='When was PUMP-101 last serviced?',
    chunks=demo_chunks,
)

print('=== Formatted LLM Prompt ===\n')
print(prompt)